# Mini-Project 2 — Part 3: SAM2 Segmentation
**Segment Anything Model 2 (Meta) on Pascal VOC 2007**

## 1. Install SAM2

In [ ]:
!pip install torch torchvision -q
# Install SAM2 from Meta
!pip install 'git+https://github.com/facebookresearch/sam2.git' -q

In [ ]:
import json, os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.datasets import VOCSegmentation
from torchvision import transforms
from torch.utils.data import DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

with open('/content/project_config.json') as f:
    cfg = json.load(f)

VOC_ROOT    = cfg['voc_root']
IMG_SIZE    = tuple(cfg['img_size'])
NUM_CLASSES = cfg['num_classes']
VOC_CLASSES = cfg['voc_classes']
SAVE_DIR    = '/content/drive/MyDrive/SHBT261_mini_proj2/mini_proj2_checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

## 2. Download SAM2 Checkpoint

In [ ]:
# Download the lightweight vit_b checkpoint (~375MB)
CKPT_DIR = '/content/sam2_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_PATH = f'{CKPT_DIR}/sam2.1_hiera_base_plus.pt'

if not os.path.exists(CKPT_PATH):
    !wget -q -O {CKPT_PATH} https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt
    print('Checkpoint downloaded.')
else:
    print('Checkpoint already present.')

## 3. Load SAM2 Model

In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

sam2_model = build_sam2(
    config_file='sam2_hiera_b+.yaml',
    ckpt_path=CKPT_PATH,
    device=DEVICE
)
predictor = SAM2ImagePredictor(sam2_model)
print('SAM2 loaded.')

## 4. Dataset (without normalization — SAM2 takes raw uint8 images)

In [ ]:
# SAM2 needs raw PIL images, not normalised tensors
# We load with minimal transforms and convert manually

transform_raw = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),  # [0,1] float
])
transform_mask = transforms.Compose([
    transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(),
])

val_set = VOCSegmentation(VOC_ROOT, year='2007', image_set='val', download=False,
                          transform=transform_raw, target_transform=transform_mask)
print(f'Test set: {len(val_set)} images')

## 5. SAM2 Zero-Shot Inference with GT Bounding Box Prompts
We provide one bounding box per class instance from the GT mask as a prompt.

In [ ]:
def get_bboxes_from_mask(mask_np, class_id):
    """Return bounding boxes for all connected regions of class_id in mask."""
    from scipy import ndimage
    binary = (mask_np == class_id).astype(np.uint8)
    labeled, n = ndimage.label(binary)
    bboxes = []
    for i in range(1, n + 1):
        rows = np.where((labeled == i).any(axis=1))[0]
        cols = np.where((labeled == i).any(axis=0))[0]
        if len(rows) > 0 and len(cols) > 0:
            bboxes.append([cols.min(), rows.min(), cols.max(), rows.max()])
    return bboxes

def predict_sam2_mask(predictor, img_tensor, gt_mask_np, num_classes):
    """Run SAM2 with GT bbox prompts per class. Returns combined pred mask."""
    # Convert image tensor (C,H,W) float [0,1] to uint8 (H,W,C)
    img_np = (img_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    predictor.set_image(img_np)

    H, W = gt_mask_np.shape
    pred_mask = np.zeros((H, W), dtype=np.uint8)  # 0 = background

    for cls_id in range(1, num_classes):  # skip background
        bboxes = get_bboxes_from_mask(gt_mask_np, cls_id)
        for bbox in bboxes:
            box = np.array(bbox, dtype=np.float32)
            masks, scores, _ = predictor.predict(
                point_coords=None, point_labels=None,
                box=box[None, :], multimask_output=False
            )
            region = masks[0].astype(bool)  # (H,W)
            pred_mask[region] = cls_id

    return pred_mask

In [ ]:
# Evaluate on a subset for speed (full eval in 04_evaluation.ipynb)
N_EVAL = 50  # increase for full evaluation
all_preds, all_gts = [], []

for i in range(N_EVAL):
    img_tensor, mask_tensor = val_set[i]
    gt_mask = mask_tensor.squeeze().numpy().astype(np.uint8)

    pred_mask = predict_sam2_mask(predictor, img_tensor, gt_mask, NUM_CLASSES)
    all_preds.append(pred_mask)
    all_gts.append(gt_mask)

    if (i + 1) % 10 == 0:
        print(f'Processed {i+1}/{N_EVAL}')

print('Inference complete.')

## 6. Quick mIoU Check

In [ ]:
def miou_numpy(preds, gts, num_classes, ignore=255):
    ious = []
    for c in range(num_classes):
        tp = fp = fn = 0
        for p, g in zip(preds, gts):
            mask = g != ignore
            tp += ((p == c) & (g == c) & mask).sum()
            fp += ((p == c) & (g != c) & mask).sum()
            fn += ((p != c) & (g == c) & mask).sum()
        denom = tp + fp + fn
        if denom > 0:
            ious.append(tp / denom)
    return np.mean(ious)

miou = miou_numpy(all_preds, all_gts, NUM_CLASSES)
print(f'SAM2 zero-shot mIoU (bbox prompts, {N_EVAL} images): {miou:.4f}')

## 7. Visualize SAM2 Predictions

In [ ]:
VOC_COLORMAP = np.array([
    [0,0,0],[128,0,0],[0,128,0],[128,128,0],[0,0,128],
    [128,0,128],[0,128,128],[128,128,128],[64,0,0],[192,0,0],
    [64,128,0],[192,128,0],[64,0,128],[192,0,128],[64,128,128],
    [192,128,128],[0,64,0],[128,64,0],[0,192,0],[128,192,0],[0,64,128]
], dtype=np.uint8)

def show_comparison(img_tensor, gt, pred, title=''):
    img = (img_tensor.permute(1,2,0).numpy() * 255).astype(np.uint8)
    gt_c  = VOC_COLORMAP[np.clip(gt,  0, 20)]
    pr_c  = VOC_COLORMAP[np.clip(pred, 0, 20)]
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(img);   ax[0].set_title('Image');      ax[0].axis('off')
    ax[1].imshow(gt_c);  ax[1].set_title('Ground Truth'); ax[1].axis('off')
    ax[2].imshow(pr_c);  ax[2].set_title(f'SAM2 Pred {title}'); ax[2].axis('off')
    plt.tight_layout(); plt.show()

for idx in [0, 5, 10]:
    img_t, _ = val_set[idx]
    show_comparison(img_t, all_gts[idx], all_preds[idx], title=f'[{idx}]')

In [ ]:
# Save predictions for use in evaluation notebook
import pickle
with open(os.path.join(SAVE_DIR, 'sam2_preds.pkl'), 'wb') as f:
    pickle.dump({'preds': all_preds, 'gts': all_gts, 'n_eval': N_EVAL}, f)
print('SAM2 predictions saved.')